In [ ]:
import gym
from gym import spaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import heapq
import copy
import sys
from datetime import timedelta

# Fixed seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

MAX_STEPS_PER_EPISODE = 3000
MAX_BATCH_SIZE = 5

# Global variables to track the best schedule found
best_jobs_not_scheduled_count_data = float('inf')
best_jobs_on_time_count_data = -1
best_makespan = float('inf')

best_schedule_data = None
best_job_completion_times_data = None
best_total_tardiness_minutes_data = 0.0
best_jobs_late_count_data = 0
best_jobs_on_time_count_data = 0
best_total_accumulated_penalty_cost_data = 0.0
best_first_job_scheduled_time_data = None
best_current_time_data = None
best_episode_num = -1

# --- Excel Data Loading ---
excel_file_name = "DQN_SHOP FLOOR DATA.xlsx"
excel_path = excel_file_name

try:
    sheets_dict = pd.read_excel(excel_path, sheet_name=None)
except FileNotFoundError:
    print(f"Error: '{excel_path}' not found.", file=sys.stderr)
    sys.exit(1)

original_jobs_df_raw = sheets_dict['Jobs_to_schedule'].copy()
original_jobs_df_raw.rename(columns={
    'Part ID': 'Part IDs',
    'Order Qty': 'Quantities to Schedule (Nos)',
    'User Due Date & Time': 'User Due Date',
    'Penalty': 'Penalty (Yes, No)',
    'Start Date & Time': 'Job Start Date',
    'Priority': 'Priority'# Added rename for the new 'Start Date & Time' column

}, inplace=True)
if 'User Due Date' in original_jobs_df_raw.columns:
    original_jobs_df_raw['User Due Date'] = pd.to_datetime(original_jobs_df_raw['User Due Date'])
if 'Job Start Date' in original_jobs_df_raw.columns:
    original_jobs_df_raw['Job Start Date'] = pd.to_datetime(original_jobs_df_raw['Job Start Date'])
if 'Order ID' in original_jobs_df_raw.columns:
    original_jobs_df_raw['Order ID'] = original_jobs_df_raw['Order ID'].astype(str).str.strip()

machines_df = pd.read_excel(excel_path, sheet_name='Machine_INFO', header=0).copy()
if 'Machine_ID' in machines_df.columns:
    machines_df.rename(columns={'Machine_ID': 'Machine ID'}, inplace=True)
if 'Machine ID' in machines_df.columns:
    machines_df['Machine ID'] = machines_df['Machine ID'].astype(str).str.strip()
machines_df = machines_df[['Machine ID', 'Machine Name']]

operators_df = pd.read_excel(excel_path, sheet_name='Operator_INFO', header=0).copy()
if 'Operator ID' in operators_df.columns:
    operators_df['Operator ID'] = operators_df['Operator ID'].astype(str).str.strip()

mc_op_mapping_df = pd.read_excel(excel_path, sheet_name='mac_op_mapping', header=0).copy()
if 'Machine_ID' in mc_op_mapping_df.columns:
    mc_op_mapping_df.rename(columns={'Machine_ID': 'Machine ID'}, inplace=True)
if 'Operator_ID' in mc_op_mapping_df.columns:
    mc_op_mapping_df.rename(columns={'Operator_ID': 'Operator ID'}, inplace=True)
if 'Machine ID' in mc_op_mapping_df.columns:
    mc_op_mapping_df['Machine ID'] = mc_op_mapping_df['Machine ID'].astype(str).str.strip()
if 'Operator ID' in mc_op_mapping_df.columns:
    mc_op_mapping_df['Operator ID'] = mc_op_mapping_df['Operator ID'].astype(str).str.strip()

fixture_info_df = pd.read_excel(excel_path, sheet_name='Fixture_INFO', header=0).copy()
if 'Fixture ID' in fixture_info_df.columns:
    fixture_info_df['Fixture ID'] = fixture_info_df['Fixture ID'].astype(str).str.strip()
if 'Quantity' not in fixture_info_df.columns:
    fixture_info_df['Quantity'] = 1

mc_fix_mapping_df = pd.read_excel(excel_path, sheet_name='Mc_fix_mapping', header=0).copy()
if 'Machine_ID' in mc_fix_mapping_df.columns:
    mc_fix_mapping_df.rename(columns={'Machine_ID': 'Machine ID'}, inplace=True)
if 'Fixture_ID' in mc_fix_mapping_df.columns:
    mc_fix_mapping_df.rename(columns={'Fixture_ID': 'Fixture ID'}, inplace=True)
if 'Machine ID' in mc_fix_mapping_df.columns:
    mc_fix_mapping_df['Machine ID'] = mc_fix_mapping_df['Machine ID'].astype(str).str.strip()
if 'Fixture ID' in mc_fix_mapping_df.columns:
    mc_fix_mapping_df['Fixture ID'] = mc_fix_mapping_df['Fixture ID'].astype(str).str.strip()

operation_df = pd.read_excel(excel_path, sheet_name='Operation', header=0).copy()
operation_df.rename(columns={
    'Part ID': 'Part IDs',
    'Setup Time (min)': 'ST (minutes)',
    'Process Time (min)': 'PT (minutes)'
}, inplace=True)
if 'Machine_ID' in operation_df.columns:
    operation_df.rename(columns={'Machine_ID': 'Machine ID'}, inplace=True)
if 'Fixture_ID' in operation_df.columns:
    operation_df.rename(columns={'Fixture_ID': 'Fixture ID'}, inplace=True)
if 'Machine ID' in operation_df.columns:
    operation_df['Machine ID'] = operation_df['Machine ID'].astype(str).str.strip()
if 'Fixture ID' in operation_df.columns:
    operation_df['Fixture ID'] = operation_df['Fixture ID'].astype(str).str.strip()

# Planned Downtime
planned_machine_downtime_df = pd.read_excel(excel_path, sheet_name='Planned_Machine_Downtime', header=0).copy()
if 'Machine_ID' in planned_machine_downtime_df.columns:
    planned_machine_downtime_df.rename(columns={'Machine_ID': 'Machine ID'}, inplace=True)
if 'Start_DateTime' in planned_machine_downtime_df.columns:
    planned_machine_downtime_df.rename(columns={'Start_DateTime': 'Start Time'}, inplace=True)
if 'End_DateTime' in planned_machine_downtime_df.columns:
    planned_machine_downtime_df.rename(columns={'End_DateTime': 'End Time'}, inplace=True)
if 'Machine ID' in planned_machine_downtime_df.columns and 'Start Time' in planned_machine_downtime_df.columns and 'End Time' in planned_machine_downtime_df.columns:
    planned_machine_downtime_df['Machine ID'] = planned_machine_downtime_df['Machine ID'].astype(str).str.strip()
    planned_machine_downtime_df['Start Time'] = pd.to_datetime(planned_machine_downtime_df['Start Time'])
    planned_machine_downtime_df['End Time'] = pd.to_datetime(planned_machine_downtime_df['End Time'])
planned_machine_downtime_df.sort_values(by='Start Time', inplace=True)

planned_operator_downtime_df = pd.read_excel(excel_path, sheet_name='Planned_Operator_Downtime', header=0).copy()
if 'Operator_ID' in planned_operator_downtime_df.columns:
    planned_operator_downtime_df.rename(columns={'Operator_ID': 'Operator ID'}, inplace=True)
if 'Start_DateTime' in planned_operator_downtime_df.columns:
    planned_operator_downtime_df.rename(columns={'Start_DateTime': 'Start Time'}, inplace=True)
if 'End_DateTime' in planned_operator_downtime_df.columns:
    planned_operator_downtime_df.rename(columns={'End_DateTime': 'End Time'}, inplace=True)
if 'Operator ID' in planned_operator_downtime_df.columns and 'Start Time' in planned_operator_downtime_df.columns and 'End Time' in planned_operator_downtime_df.columns:
    planned_operator_downtime_df['Operator ID'] = planned_operator_downtime_df['Operator ID'].astype(str).str.strip()
    planned_operator_downtime_df['Start Time'] = pd.to_datetime(planned_operator_downtime_df['Start Time'])
    planned_operator_downtime_df['End Time'] = pd.to_datetime(planned_operator_downtime_df['End Time'])
planned_operator_downtime_df.sort_values(by='Start Time', inplace=True)

planned_machine_downtime = []
if 'Machine ID' in planned_machine_downtime_df.columns and 'Start Time' in planned_machine_downtime_df.columns and 'End Time' in planned_machine_downtime_df.columns:
    planned_machine_downtime = planned_machine_downtime_df.rename(columns={'Machine ID': 'resource_id', 'Start Time': 'start_time', 'End Time': 'end_time'}).to_dict(orient='records')

planned_operator_downtime = []
if 'Operator ID' in planned_operator_downtime_df.columns and 'Start Time' in planned_operator_downtime_df.columns and 'End Time' in planned_operator_downtime_df.columns:
    planned_operator_downtime = planned_operator_downtime_df.rename(columns={'Operator ID': 'resource_id', 'Start Time': 'start_time', 'End Time': 'end_time'}).to_dict(orient='records')

machine_list = machines_df['Machine ID'].dropna().unique().tolist() if 'Machine ID' in machines_df.columns else []
operator_list = operators_df['Operator ID'].dropna().unique().tolist() if 'Operator ID' in operators_df.columns else []
fixture_list = fixture_info_df['Fixture ID'].dropna().unique().tolist() if 'Fixture ID' in fixture_info_df.columns else []

if not machine_list: machine_list = ['M0']
if not operator_list: operator_list = ['O0']
if not fixture_list: fixture_list = ['F0']

fixture_total_quantities = {}
if 'Fixture ID' in fixture_info_df.columns and 'Quantity' in fixture_info_df.columns:
    fixture_total_quantities = dict(zip(fixture_info_df['Fixture ID'], fixture_info_df['Quantity']))
else:
    for f_id in fixture_list:
        fixture_total_quantities[f_id] = 1

MAX_QUANTITY = original_jobs_df_raw['Quantities to Schedule (Nos)'].max() if 'Quantities to Schedule (Nos)' in original_jobs_df_raw.columns and not original_jobs_df_raw.empty else 1.0
MAX_PRIORITY = original_jobs_df_raw['Priority'].max() if 'Priority' in original_jobs_df_raw.columns and not original_jobs_df_raw.empty else 10.0
MAX_SIM_MINUTES = 5000.0

print("\nData loaded.", file=sys.stdout)
sys.stdout.flush()

# --- Batching Function ---
def generate_batches(original_jobs_df_input):
    """Splits original jobs into smaller batches."""
    batched_jobs = []
    batch_to_original_job_map = {}
    original_job_num_batches = {}

    for _, original_job_data in original_jobs_df_input.iterrows():
        original_order_id = str(original_job_data['Order ID'])
        original_part_id = str(original_job_data['Part IDs'])
        total_qty = int(original_job_data['Quantities to Schedule (Nos)'])

        num_batches = (total_qty + MAX_BATCH_SIZE - 1) // MAX_BATCH_SIZE
        original_job_num_batches[original_order_id] = num_batches

        for i in range(num_batches):
            batch_qty = min(MAX_BATCH_SIZE, total_qty - i * MAX_BATCH_SIZE)
            batch = original_job_data.copy().to_dict()

            batch["Quantities to Schedule (Nos)"] = batch_qty
            batch["Batch ID"] = f"{original_order_id}_B{i+1}"
            batch["Original Order ID"] = original_order_id
            batch["Original Part ID"] = original_part_id
            batch["Original Quantity"] = total_qty

            batched_jobs.append(batch)
            batch_to_original_job_map[batch["Batch ID"]] = original_order_id

    batched_jobs_df = pd.DataFrame(batched_jobs)
    batched_jobs_df.rename(columns={'Order ID': 'Original Order ID'}, inplace=True)
    batched_jobs_df['Order ID'] = batched_jobs_df['Batch ID']

    return batched_jobs_df, batch_to_original_job_map, original_job_num_batches

batched_jobs_df_global, batch_to_original_job_map_global, original_job_num_batches_global = generate_batches(original_jobs_df_raw)
batched_jobs_df_global = batched_jobs_df_global.loc[:, ~batched_jobs_df_global.columns.duplicated()]

print("Batched Jobs loaded.", file=sys.stdout)
sys.stdout.flush()

# --- Shop Floor Environment ---
class ShopFloorEnv(gym.Env):
    def __init__(self, debug_feasibility_checks=False):
        super(ShopFloorEnv, self).__init__()

        self.original_jobs_df = original_jobs_df_raw.copy()
        self.jobs = batched_jobs_df_global.copy()
        self.batch_to_original_job_map = batch_to_original_job_map_global.copy()
        self.original_job_num_batches = original_job_num_batches_global.copy()

        self.debug_feasibility_checks = debug_feasibility_checks

        self.operation_df = operation_df.copy()
        self.machines = machines_df.copy()
        self.operators = operators_df.copy()
        self.mc_op_mapping = mc_op_mapping_df.copy()
        self.mc_fix_mapping = mc_fix_mapping_df.copy()
        self.fixture_info = fixture_info_df.copy()
        self.planned_machine_downtime = planned_machine_downtime
        self.planned_operator_downtime = planned_operator_downtime

        self.fixture_total_quantities = fixture_total_quantities.copy()
        self.fixture_available_quantities = {}
        self.active_fixture_usages = []

        self.num_jobs = len(self.jobs)
        self.num_options_per_job = self.operation_df['Option'].max() if not self.operation_df.empty else 1
        self.machine_list = machine_list
        self.operator_list = operator_list
        self.fixture_list = fixture_list
        self.num_machines = len(self.machine_list)
        self.num_operators = len(self.operator_list)
        self.num_fixtures = len(self.fixture_list)
        self.num_original_jobs = len(self.original_jobs_df)

        self.action_space = spaces.Discrete(self.num_jobs * self.num_options_per_job)

        # Observation space: job features (is_pending, quantity, priority, lateness_norm), current time,
        # machine availability, operator availability, fixture availability, original job batch completion.
        self.observation_space = spaces.Box(
            low=0.0, high=1.0,
            shape=(self.num_jobs * 4 + 1 + self.num_machines + self.num_operators + self.num_fixtures + self.num_original_jobs,),
            dtype=np.float32
        )

        self.job_committed_option = {}
        self.job_op_sequences = {}
        self._preprocess_job_op_sequences()

        self.MAX_WAIT_STEPS = 500
        self.failed_action_counts = {}
        self.last_machine_op_info = {m_id: None for m_id in self.machine_list}
        self.TIME_THRESHOLD_FOR_SETUP_SKIP_MINUTES = 10

        self.reset()

    def _preprocess_job_op_sequences(self):
        """Pre-processes operation sequences for faster lookup."""
        self.job_op_sequences = {}
        for _, original_job_data in self.original_jobs_df.iterrows():
            part_id = original_job_data['Part IDs']
            self.job_op_sequences[part_id] = {}
            for option_idx in range(self.num_options_per_job):
                ops_for_part_option = self.operation_df[
                    (self.operation_df['Part IDs'] == part_id) &
                    (self.operation_df['Option'] == option_idx + 1)
                ].copy()
                if not ops_for_part_option.empty:
                    ops_for_part_option['op_num'] = ops_for_part_option['Operation'].str.extract('(\\d+)').astype(int)
                    ops_for_part_option.sort_values(by='op_num', inplace=True)
                    self.job_op_sequences[part_id][option_idx] = ops_for_part_option.to_dict(orient='records')
                else:
                    self.job_op_sequences[part_id][option_idx] = []

    def reset(self):
        """Resets the environment for a new episode."""
        self.jobs, self.batch_to_original_job_map, self.original_job_num_batches = generate_batches(self.original_jobs_df)
        self.jobs = self.jobs.loc[:, ~self.jobs.columns.duplicated()]
        self.num_jobs = len(self.jobs)

        self.remaining_jobs = set(range(self.num_jobs))
        self.schedule = []

        self.original_job_completion_status = {
            original_order_id: set() for original_order_id in self.original_jobs_df['Order ID'].unique()
        }
        self.original_job_completion_times = {}

        self.job_committed_option = {}
        self.job_last_op_finish_time = {}

        if 'Job Start Date' in self.original_jobs_df.columns and not self.original_jobs_df['Job Start Date'].empty:
            earliest_start = pd.to_datetime(self.original_jobs_df['Job Start Date'].min())
        else:
            earliest_start = pd.Timestamp('2024-07-01 00:00:00')
        self.initial_simulation_start_time = earliest_start
        self.current_time = earliest_start
        self.first_job_scheduled_time = None

        self.machine_total_processing_time = {m_id: 0.0 for m_id in self.machine_list}
        self.operator_total_processing_time = {o_id: 0.0 for o_id in self.operator_list}
        self.total_tardiness_minutes = 0.0
        self.jobs_on_time_count = 0
        self.jobs_late_count = 0
        self.total_accumulated_penalty_cost = 0.0

        self.machine_available_time = {m: self.current_time for m in self.machine_list}
        for downtime_event in self.planned_machine_downtime:
            resource_id = downtime_event['resource_id']
            start_time = pd.to_datetime(downtime_event['start_time'])
            end_time = pd.to_datetime(downtime_event['end_time'])
            if resource_id in self.machine_available_time:
                if start_time <= self.current_time < end_time:
                    self.machine_available_time[resource_id] = max(self.machine_available_time[resource_id], end_time)

        self.operator_available_time = {o: self.current_time for o in self.operator_list}
        for downtime_event in self.planned_operator_downtime:
            resource_id = downtime_event['resource_id']
            start_time = pd.to_datetime(downtime_event['start_time'])
            end_time = pd.to_datetime(downtime_event['end_time'])
            if resource_id in self.operator_available_time:
                if start_time <= self.current_time < end_time:
                    self.operator_available_time[resource_id] = max(self.operator_available_time[resource_id], end_time)

        self.fixture_available_quantities = self.fixture_total_quantities.copy()
        self.active_fixture_usages = []

        self.wait_count = 0
        self.failed_action_counts = {}
        self.last_machine_op_info = {m_id: None for m_id in self.machine_list}

        return self._get_obs()

    def _release_completed_fixtures(self, current_time):
        """Releases fixtures that are no longer in use."""
        for usage in self.active_fixture_usages[:]:
            if usage["end_time"] <= current_time:
                fixture_id = usage["fixture_id"]
                if fixture_id in self.fixture_available_quantities:
                    self.fixture_available_quantities[fixture_id] += 1
                else:
                    self.fixture_available_quantities[fixture_id] = 1
                self.active_fixture_usages.remove(usage)

    def _get_operator_for_machine(self, machine_id):
        """Finds a suitable operator for a given machine."""
        if pd.isna(machine_id) or machine_id is None:
            return None
        machine_id_str = str(machine_id).strip()
        mapping = self.mc_op_mapping[self.mc_op_mapping['Machine ID'] == machine_id_str]
        if not mapping.empty and 'Operator ID' in mapping.columns:
            possible_operators = mapping['Operator ID'].dropna().unique().tolist()
            possible_operators = [op for op in possible_operators if str(op).lower() != 'nan']
            if possible_operators:
                return random.choice(possible_operators)
        if 'O0' in self.operator_list:
            return 'O0'
        return None

    def _get_obs(self):
        """Constructs the observation (state) for the agent."""
        obs = []

        # Batch-specific info: pending, quantity, priority, lateness
        for i in range(self.num_jobs):
            is_pending = 1.0 if i in self.remaining_jobs else 0.0
            batch_data = self.jobs.iloc[i]

            quantity = batch_data.get('Quantities to Schedule (Nos)', 0) / MAX_QUANTITY
            priority = batch_data.get('Priority', 1) / MAX_PRIORITY

            due_date_col = 'User Due Date'
            arrival_time = pd.to_datetime(batch_data.get("Job Start Date", self.current_time))
            effective_time_for_lateness = max(self.current_time, arrival_time)

            original_order_id = batch_data['Original Order ID']
            completion_or_current_time = self.original_job_completion_times.get(original_order_id, effective_time_for_lateness)

            if due_date_col in batch_data and pd.notna(batch_data[due_date_col]):
                due_date = pd.to_datetime(batch_data[due_date_col])
                delay_timedelta = pd.Timedelta(completion_or_current_time - due_date)
                lateness = max(pd.Timedelta(0), delay_timedelta).total_seconds() / 60
            else:
                lateness = 0
            lateness_norm = lateness / MAX_SIM_MINUTES

            obs.extend([is_pending, quantity, priority, lateness_norm])

        # Current simulation time
        current_time_timedelta = pd.Timedelta(self.current_time - pd.Timestamp('2024-07-01 00:00:00'))
        current_time_minutes = current_time_timedelta.total_seconds() / 60
        obs.append(current_time_minutes / MAX_SIM_MINUTES)

        # Resource availability
        for mid in self.machine_list:
            time_until_available_timedelta = pd.Timedelta(self.machine_available_time.get(mid, self.current_time) - self.current_time)
            time_until_available = max(pd.Timedelta(0), time_until_available_timedelta).total_seconds() / 60
            obs.append(time_until_available / MAX_SIM_MINUTES)

        for oid in self.operator_list:
            time_until_available_timedelta = pd.Timedelta(self.operator_available_time.get(oid, self.current_time) - self.current_time)
            time_until_available = max(pd.Timedelta(0), time_until_available_timedelta).total_seconds() / 60
            obs.append(time_until_available / MAX_SIM_MINUTES)

        # Fixture availability (normalized quantity)
        for fid in self.fixture_list:
            total_qty = self.fixture_total_quantities.get(fid, 1)
            available_qty = self.fixture_available_quantities.get(fid, 0)
            fixture_avail_norm = available_qty / total_qty
            obs.append(fixture_avail_norm)

        # Original Job Batch Completion Awareness
        for original_order_id in self.original_jobs_df['Order ID'].unique():
            batches_completed = len(self.original_job_completion_status.get(original_order_id, set()))
            total_batches = self.original_job_num_batches.get(original_order_id, 1)
            normalized_batches_remaining = (total_batches - batches_completed) / total_batches
            obs.append(normalized_batches_remaining)

        return np.array(obs, dtype=np.float32)

    def _increment_failed_action(self, job_id, option_id):
        """Tracks how many times an action has failed."""
        key = (job_id, option_id)
        self.failed_action_counts[key] = self.failed_action_counts.get(key, 0) + 1

    def _has_failed_too_many_times(self, job_id, option_id):
        """Checks if an action has failed too many times."""
        key = (job_id, option_id)
        return self.failed_action_counts.get(key, 0) > 3

    def _is_action_feasible(self, job_id, option_idx, current_sim_time):
        """Checks if an action is structurally and immediately feasible."""
        batch_data = self.jobs.iloc[job_id]
        part_id_for_ops = batch_data['Part IDs']
        ops_for_part_option = self.job_op_sequences[part_id_for_ops][option_idx]

        if not ops_for_part_option:
            return False

        first_op_data = ops_for_part_option[0]
        assigned_fixture_first_op = str(first_op_data.get('Fixture ID')).strip() if pd.notna(first_op_data.get('Fixture ID')) else None
        if assigned_fixture_first_op == "nan": assigned_fixture_first_op = None

        if assigned_fixture_first_op is None or assigned_fixture_first_op not in self.fixture_list:
            return False

        if self.fixture_available_quantities.get(assigned_fixture_first_op, 0) <= 0:
            return False

        for op_data in ops_for_part_option:
            operation_name = op_data['Operation']

            assigned_machine = str(op_data['Machine ID']).strip() if pd.notna(op_data.get('Machine ID')) else None
            if assigned_machine == "nan": assigned_machine = None
            if assigned_machine is None or assigned_machine not in self.machine_list:
                return False

            assigned_fixture = str(op_data['Fixture ID']).strip() if pd.notna(op_data.get('Fixture ID')) else None
            if assigned_fixture == "nan": assigned_fixture = None
            if assigned_fixture is None or assigned_fixture not in self.fixture_list:
                return False

            if assigned_fixture not in self.fixture_total_quantities:
                return False

            quantity = batch_data.get('Quantities to Schedule (Nos)', 1)
            st_time = float(op_data.get('ST (minutes)', 0)) if pd.notna(op_data.get('ST (minutes)')) else 0.0
            pt_time = float(op_data.get('PT (minutes)', 0))
            process_time_for_feasibility = st_time + (pt_time * quantity)

            if process_time_for_feasibility <= 0.0:
                return False

            assigned_operator = self._get_operator_for_machine(assigned_machine)
            if assigned_operator is None or assigned_operator not in self.operator_list:
                return False

            valid_mc_op_mapping = self.mc_op_mapping[
                (self.mc_op_mapping['Machine ID'] == assigned_machine) &
                (self.mc_op_mapping['Operator ID'] == assigned_operator)
            ]
            if valid_mc_op_mapping.empty:
                return False

            valid_mc_fix_mapping = self.mc_fix_mapping[
                (self.mc_fix_mapping['Machine ID'] == assigned_machine) &
                (self.mc_fix_mapping['Fixture ID'] == assigned_fixture)
            ]
            if valid_mc_fix_mapping.empty:
                return False

        return True

    def _schedule_batch_operations(self, batch_idx, option_idx, batch_base_start_time,
                                   temp_machine_availability, temp_operator_availability,
                                   temp_fixture_available_quantities, temp_active_fixture_usages_in_step,
                                   machines_locked_at_current_time, operators_locked_at_current_time,
                                   fixtures_locked_at_current_time, initial_current_time, is_primary_action=False):
        """Attempts to schedule a batch's entire operation sequence."""
        batch_data = self.jobs.iloc[batch_idx]
        part_id_for_ops = batch_data['Part IDs']
        ops_to_schedule = self.job_op_sequences[part_id_for_ops][option_idx]

        last_op_finish_time = batch_base_start_time
        reward_from_ops = 0.0

        batch_arrival_time = pd.to_datetime(batch_data.get("Job Start Date", initial_current_time))

        for op_data in ops_to_schedule:
            operation_name = op_data['Operation']
            assigned_machine = str(op_data['Machine ID']).strip()
            assigned_fixture = str(op_data['Fixture ID']).strip()
            quantity = batch_data.get("Quantities to Schedule (Nos)", 1)
            st_time = float(op_data.get('ST (minutes)', 0))
            pt_time = float(op_data.get('PT (minutes)', 0))

            assigned_operator = self._get_operator_for_machine(assigned_machine)
            if assigned_operator is None:
                return False, initial_current_time, 0.0

            # Setup time logic
            actual_st_time = st_time
            last_op_info_on_machine = self.last_machine_op_info.get(assigned_machine)
            effective_check_start_time = max(last_op_finish_time,
                                             temp_machine_availability.get(assigned_machine, initial_current_time),
                                             temp_operator_availability.get(assigned_operator, initial_current_time),
                                             batch_arrival_time)

            if last_op_info_on_machine:
                prev_part_id = last_op_info_on_machine['part_id']
                prev_option_idx = last_op_info_on_machine['option_idx']
                prev_finish_time = last_op_info_on_machine['finish_time']

                time_since_last_op_minutes = (effective_check_start_time - prev_finish_time).total_seconds() / 60

                if (part_id_for_ops == prev_part_id and
                    option_idx == prev_option_idx and
                    time_since_last_op_minutes <= self.TIME_THRESHOLD_FOR_SETUP_SKIP_MINUTES):
                    actual_st_time = 0.0
                    reward_from_ops += 10.0

            process_time = actual_st_time + (pt_time * quantity)

            if temp_fixture_available_quantities.get(assigned_fixture, 0) <= 0:
                return False, initial_current_time, 0.0

            current_op_start_candidate = max(last_op_finish_time,
                                             temp_machine_availability.get(assigned_machine, initial_current_time),
                                             temp_operator_availability.get(assigned_operator, initial_current_time),
                                             batch_arrival_time)

            # Enforce planned downtime
            downtime_check_needed = True
            while downtime_check_needed:
                downtime_check_needed = False
                for downtime_event in self.planned_machine_downtime:
                    if downtime_event['resource_id'] == assigned_machine:
                        dt_start = downtime_event['start_time']
                        dt_end = downtime_event['end_time']
                        if dt_start <= current_op_start_candidate < dt_end:
                            current_op_start_candidate = dt_end
                            downtime_check_needed = True
                            break

            downtime_check_needed_operator = True
            while downtime_check_needed_operator:
                downtime_check_needed_operator = False
                for downtime_event in self.planned_operator_downtime:
                    if downtime_event['resource_id'] == assigned_operator:
                        dt_start = downtime_event['start_time']
                        dt_end = downtime_event['end_time']
                        if dt_start <= current_op_start_candidate < dt_end:
                            current_op_start_candidate = dt_end
                            downtime_check_needed_operator = True
                            break

            op_start_time = current_op_start_candidate

            if op_start_time == initial_current_time and not is_primary_action:
                if assigned_machine in machines_locked_at_current_time or \
                   assigned_operator in operators_locked_at_current_time or \
                   assigned_fixture in fixtures_locked_at_current_time:
                    return False, initial_current_time, 0.0

            op_finish_time = op_start_time + pd.Timedelta(minutes=process_time)

            temp_fixture_available_quantities[assigned_fixture] -= 1
            temp_active_fixture_usages_in_step.append({
                'fixture_id': assigned_fixture,
                'end_time': op_finish_time
            })

            self.schedule.append({
                'job_id': batch_idx,
                'original_order_id': batch_data['Original Order ID'],
                'original_part_id': batch_data['Original Part ID'],
                'part_id': part_id_for_ops,
                'batch_id': batch_data['Batch ID'],
                'option_idx': option_idx,
                'operation_name': operation_name,
                'assigned_machine': assigned_machine,
                'assigned_operator': assigned_operator,
                'assigned_fixture': assigned_fixture,
                'process_time': process_time,
                'start_time': op_start_time,
                'finish_time': op_finish_time,
                'batch_data': batch_data
            })

            temp_machine_availability[assigned_machine] = op_finish_time
            temp_operator_availability[assigned_operator] = op_finish_time

            if op_start_time == initial_current_time:
                machines_locked_at_current_time.add(assigned_machine)
                operators_locked_at_current_time.add(assigned_operator)
                fixtures_locked_at_current_time.add(assigned_fixture)

            last_op_finish_time = op_finish_time

        return True, last_op_finish_time, reward_from_ops

    def calculate_true_machine_idle_time(self, initial_current_time, max_finish_time_in_this_step):
        """Calculates machine idle time within a step."""
        total_idle_time = 0.0
        for machine_id in self.machine_list:
            last_available_time = self.machine_available_time.get(machine_id, initial_current_time)
            if last_available_time < max_finish_time_in_this_step:
                idle_timedelta = pd.Timedelta(max_finish_time_in_this_step - last_available_time)
                idle_duration = (max(pd.Timedelta(0), idle_timedelta)).total_seconds() / 60
                total_idle_time += idle_duration
        return total_idle_time

    def calculate_true_operator_idle_time(self, start_time, end_time):
        """Calculates operator idle time within a step."""
        total_idle = 0.0
        for oid in self.operator_list:
            available_time = self.operator_available_time.get(oid, start_time)
            if available_time <= start_time:
                op_idle_timedelta = pd.Timedelta(end_time - start_time)
                idle_duration = (max(pd.Timedelta(0), op_idle_timedelta)).total_seconds() / 60
                total_idle += idle_duration
        return total_idle

    def step(self, action):
        """Performs one step in the environment based on the agent's action."""
        initial_current_time = self.current_time
        self._release_completed_fixtures(initial_current_time)
        total_reward_for_this_step = 0.0

        temp_machine_availability = self.machine_available_time.copy()
        temp_operator_availability = self.operator_available_time.copy()
        temp_fixture_available_quantities = self.fixture_available_quantities.copy()
        temp_active_fixture_usages_in_step = []

        machines_locked_at_current_time = set()
        operators_locked_at_current_time = set()
        fixtures_locked_at_current_time = set()

        batches_whose_operations_started_in_this_step = set()
        batch_final_completion_times_in_this_step = {}
        max_finish_time_across_all_ops_in_step = initial_current_time

        # Process primary action
        primary_action_index = action
        primary_batch_idx = primary_action_index // self.num_options_per_job
        primary_option_idx = primary_action_index % self.num_options_per_job

        if action == -1:
            pass
        else:
            pass

        if primary_batch_idx not in self.remaining_jobs:
            self.wait_count += 1
            self._increment_failed_action(primary_batch_idx, primary_option_idx)
            done = len(self.remaining_jobs) == 0 or self.wait_count > self.MAX_WAIT_STEPS
            return self._get_obs(), -100.0, done, {}

        if not self._is_action_feasible(primary_batch_idx, primary_option_idx, self.current_time):
            self.wait_count += 1
            self._increment_failed_action(primary_batch_idx, primary_option_idx)
            done = len(self.remaining_jobs) == 0 or self.wait_count > self.MAX_WAIT_STEPS
            return self._get_obs(), -500.0, done, {}

        primary_batch_data = self.jobs.iloc[primary_batch_idx]
        primary_batch_release_time = pd.to_datetime(primary_batch_data.get("Job Start Date", initial_current_time))
        primary_batch_base_start_time = primary_batch_release_time

        primary_batch_scheduled, primary_batch_overall_finish_time, primary_op_reward = self._schedule_batch_operations(
            primary_batch_idx, primary_option_idx, primary_batch_base_start_time,
            temp_machine_availability, temp_operator_availability,
            temp_fixture_available_quantities, temp_active_fixture_usages_in_step,
            machines_locked_at_current_time, operators_locked_at_current_time,
            fixtures_locked_at_current_time, initial_current_time, is_primary_action=True
        )
        total_reward_for_this_step += primary_op_reward

        if not primary_batch_scheduled:
             self.wait_count += 1
             self._increment_failed_action(primary_batch_idx, primary_option_idx)
             done = len(self.remaining_jobs) == 0 or self.wait_count > self.MAX_WAIT_STEPS
             return self._get_obs(), -500.0, done, {}

        self.job_committed_option[primary_batch_idx] = primary_option_idx
        batch_final_completion_times_in_this_step[primary_batch_idx] = primary_batch_overall_finish_time
        batches_whose_operations_started_in_this_step.add(primary_batch_idx)
        max_finish_time_across_all_ops_in_step = max(max_finish_time_across_all_ops_in_step, primary_batch_overall_finish_time)

        # Greedy parallel scheduling
        eligible_remaining_batches = [
            j for j in self.remaining_jobs if j not in batches_whose_operations_started_in_this_step
        ]

        temp_eligible_batches_df = self.jobs.iloc[eligible_remaining_batches].copy()
        if 'Job Start Date' in temp_eligible_batches_df.columns and 'Priority' in temp_eligible_batches_df.columns:
            temp_eligible_batches_df.sort_values(by=['Job Start Date', 'Priority'], ascending=[True, False], inplace=True)
        elif 'Job Start Date' in temp_eligible_batches_df.columns:
            temp_eligible_batches_df.sort_values(by='Job Start Date', ascending=True, inplace=True)
        elif 'Priority' in temp_eligible_batches_df.columns:
            temp_eligible_batches_df.sort_values(by='Priority', ascending=False, inplace=True)

        for candidate_batch_idx in temp_eligible_batches_df.index:
            if candidate_batch_idx in batches_whose_operations_started_in_this_step:
                continue

            candidate_batch_data = self.jobs.iloc[candidate_batch_idx]
            candidate_batch_release_time = pd.to_datetime(candidate_batch_data.get("Job Start Date", initial_current_time))

            if self.current_time < candidate_batch_release_time:
                continue

            candidate_batch_base_start_time = candidate_batch_release_time

            candidate_batch_scheduled = False
            candidate_batch_overall_finish_time = initial_current_time
            parallel_op_reward = 0.0

            if candidate_batch_idx in self.job_committed_option:
                cand_option_idx_to_try = self.job_committed_option[candidate_batch_idx]
                if self._is_action_feasible(candidate_batch_idx, cand_option_idx_to_try, self.current_time):
                    candidate_batch_scheduled, candidate_batch_overall_finish_time, parallel_op_reward = self._schedule_batch_operations(
                        candidate_batch_idx, cand_option_idx_to_try, candidate_batch_base_start_time,
                        temp_machine_availability, temp_operator_availability,
                        temp_fixture_available_quantities, temp_active_fixture_usages_in_step,
                        machines_locked_at_current_time, operators_locked_at_current_time,
                        fixtures_locked_at_current_time, initial_current_time, is_primary_action=False
                    )
            else:
                options_to_try = list(range(self.num_options_per_job))
                random.shuffle(options_to_try)

                for cand_option_idx in options_to_try:
                    if self._is_action_feasible(candidate_batch_idx, cand_option_idx, self.current_time):
                        candidate_batch_scheduled, candidate_batch_overall_finish_time, parallel_op_reward = self._schedule_batch_operations(
                            candidate_batch_idx, cand_option_idx, candidate_batch_base_start_time,
                            temp_machine_availability, temp_operator_availability,
                            temp_fixture_available_quantities, temp_active_fixture_usages_in_step,
                            machines_locked_at_current_time, operators_locked_at_current_time,
                            fixtures_locked_at_current_time, initial_current_time, is_primary_action=False
                        )
                        if candidate_batch_scheduled:
                            self.job_committed_option[candidate_batch_idx] = cand_option_idx
                            break

            if candidate_batch_scheduled:
                batch_final_completion_times_in_this_step[candidate_batch_idx] = candidate_batch_overall_finish_time
                batches_whose_operations_started_in_this_step.add(candidate_batch_idx)
                max_finish_time_across_all_ops_in_step = max(max_finish_time_across_all_ops_in_step, candidate_batch_overall_finish_time)
                total_reward_for_this_step += parallel_op_reward

        # Update first job scheduled time
        for op in self.schedule[-len(batches_whose_operations_started_in_this_step):]:
            if self.first_job_scheduled_time is None or op['start_time'] < self.first_job_scheduled_time:
                self.first_job_scheduled_time = op['start_time']

        # Time advancement
        if batches_whose_operations_started_in_this_step:
            self.current_time = max(self.current_time, max_finish_time_across_all_ops_in_step)
            self.wait_count = 0
        else:
            next_times = []
            for mc_time in self.machine_available_time.values():
                if mc_time > self.current_time:
                    next_times.append(mc_time)
            for op_time in self.operator_available_time.values():
                if op_time > self.current_time:
                    next_times.append(op_time)
            for usage in self.active_fixture_usages:
                if usage['end_time'] > self.current_time:
                    next_times.append(usage['end_time'])
            for batch_idx in self.remaining_jobs:
                batch_data = self.jobs.iloc[batch_idx]
                arrival_time = pd.to_datetime(batch_data.get("Job Start Date", self.current_time))
                if arrival_time > self.current_time:
                    next_times.append(arrival_time)

            if next_times:
                self.current_time = min(next_times)
            else:
                self.current_time += timedelta(minutes=10)

            total_reward_for_this_step -= (5.0 * self.wait_count)
            self.wait_count += 1

        # Check watchdog counter
        done = len(self.remaining_jobs) == 0 or self.wait_count > self.MAX_WAIT_STEPS

        # Apply temporary resource updates
        self.machine_available_time = temp_machine_availability
        self.operator_available_time = temp_operator_availability
        self.fixture_available_quantities = temp_fixture_available_quantities
        self.active_fixture_usages.extend(temp_active_fixture_usages_in_step)

        # Update last machine operation info
        for op in self.schedule[-len(batches_whose_operations_started_in_this_step):]:
            self.last_machine_op_info[op['assigned_machine']] = {
                'part_id': op['part_id'],
                'option_idx': op['option_idx'],
                'finish_time': op['finish_time']
            }

        # Calculate reward and update metrics for original jobs
        for batch_idx_in_step, finish_time_for_batch in batch_final_completion_times_in_this_step.items():
            self.remaining_jobs.discard(batch_idx_in_step)

            batch_data_for_reward = self.jobs.iloc[batch_idx_in_step]
            original_order_id = batch_data_for_reward['Original Order ID']

            self.original_job_completion_status[original_order_id].add(batch_idx_in_step)

            num_batches_for_original_job = self.original_job_num_batches[original_order_id]
            if len(self.original_job_completion_status[original_order_id]) == num_batches_for_original_job:
                all_ops_for_original_job = [
                    op for op in self.schedule if op['original_order_id'] == original_order_id
                ]
                if all_ops_for_original_job:
                    original_job_finish_time = max(op['finish_time'] for op in all_ops_for_original_job)
                else:
                    original_job_finish_time = self.current_time

                self.original_job_completion_times[original_order_id] = original_job_finish_time

                original_job_data = self.original_jobs_df[self.original_jobs_df['Order ID'] == original_order_id].iloc[0]
                priority = original_job_data.get('Priority', 1.0)

                priority_reward_component = (priority / MAX_PRIORITY) * 50.0
                total_reward_for_this_step += priority_reward_component

                due_date_col = 'User Due Date'
                if due_date_col in original_job_data and pd.notna(original_job_data[due_date_col]):
                    due_date = pd.to_datetime(original_job_data[due_date_col])
                else:
                    due_date = pd.Timestamp('2025-12-31 00:00:00')

                delay_timedelta = pd.Timedelta(original_job_finish_time - due_date)
                delay_minutes = max(0, delay_timedelta.total_seconds() / 60)
                penalty_flag = str(original_job_data.get('Penalty (Yes, No)', 'Yes')).lower()

                if penalty_flag == 'yes':
                    if delay_minutes == 0:
                        total_reward_for_this_step += 100.0
                        self.jobs_on_time_count += 1
                    else:
                        delay_penalty_component = (delay_minutes * 1.0)
                        total_reward_for_this_step -= delay_penalty_component
                        self.total_accumulated_penalty_cost += delay_penalty_component
                        self.jobs_late_count += 1
                else:
                    total_reward_for_this_step += 10.0
                    if delay_minutes == 0:
                        self.jobs_on_time_count += 1
                    else:
                        self.jobs_late_count += 1
                self.total_tardiness_minutes += delay_minutes

        # Idle penalty for unused resources
        idle_penalty_value = 0.0
        duration_of_this_step_timedelta = pd.Timedelta(max_finish_time_across_all_ops_in_step - initial_current_time)
        duration_of_this_step_minutes = duration_of_this_step_timedelta.total_seconds() / 60
        if duration_of_this_step_minutes > 0:
            for mid in self.machine_list:
                if self.machine_available_time.get(mid, initial_current_time) <= initial_current_time and \
                   mid not in machines_locked_at_current_time:
                    idle_penalty_value += 1
            for oid in self.operator_list:
                if self.operator_available_time.get(oid, initial_current_time) <= initial_current_time and \
                   oid not in operators_locked_at_current_time:
                    idle_penalty_value += 1
            for fid in self.fixture_list:
                if self.fixture_available_quantities.get(fid, 0) == self.fixture_total_quantities.get(fid, 1) and \
                   fid not in fixtures_locked_at_current_time:
                    idle_penalty_value += 1

            idle_resource_penalty_component = (idle_penalty_value * duration_of_this_step_minutes * 0.0005)
            total_reward_for_this_step -= idle_resource_penalty_component
            self.total_accumulated_penalty_cost += idle_resource_penalty_component

        true_idle_time = self.calculate_true_machine_idle_time(initial_current_time, max_finish_time_across_all_ops_in_step)
        true_idle_time_penalty_component = true_idle_time * 0.0002
        total_reward_for_this_step -= true_idle_time_penalty_component
        self.total_accumulated_penalty_cost += true_idle_time_penalty_component

        true_operator_idle_time = self.calculate_true_operator_idle_time(initial_current_time, max_finish_time_across_all_ops_in_step)
        true_operator_idle_penalty_component = true_operator_idle_time * 0.001
        total_reward_for_this_step -= true_operator_idle_penalty_component
        self.total_accumulated_penalty_cost += true_operator_idle_penalty_component

        all_original_jobs_completed = (len(self.original_job_completion_times) == len(self.original_jobs_df))
        done = all_original_jobs_completed or self.wait_count > self.MAX_WAIT_STEPS

        obs = self._get_obs()

        # Makespan penalty at episode end
        if done:
            if self.first_job_scheduled_time is None:
                self.first_job_scheduled_time = self.initial_simulation_start_time
            final_makespan_timedelta = pd.Timedelta(self.current_time - self.first_job_scheduled_time)
            final_makespan_minutes = final_makespan_timedelta.total_seconds() / 60
            makespan_penalty = final_makespan_minutes * 0.1
            total_reward_for_this_step -= makespan_penalty

        return obs, total_reward_for_this_step, done, {}

    def render(self, mode='human'):
        """Prints a detailed schedule."""
        if mode == 'human':
            print("\n--- Scheduled Operations (Granular Schedule) ---", file=sys.stdout)
            sys.stdout.flush()
            for entry in self.schedule:
                start_str = entry['start_time'].strftime('%Y-%m-%d %H:%M') if pd.notna(entry['start_time']) else 'N/A'
                finish_str = entry['finish_time'].strftime('%Y-%m-%d %H:%M') if pd.notna(entry['finish_time']) else 'N/A'
                print(f"Batch {entry['batch_id']} (Original Order {entry['original_order_id']}, Part {entry['original_part_id']}) "
                      f"Op {entry['operation_name']} (Option {entry['option_idx'] + 1}): "
                      f"Machine={entry['assigned_machine']}, Operator={entry['assigned_operator']}, "
                      f"Fixture={entry['assigned_fixture']}, PT={entry['process_time']:.2f} min, "
                      f"Start={start_str}, "
                      f"Finish={finish_str}", file=sys.stdout)
            sys.stdout.flush()
            print(f"Total operations scheduled: {len(self.schedule)}", file=sys.stdout)
            print(f"Final simulation time: {self.current_time.strftime('%Y-%m-%d %H:%M')}", file=sys.stdout)
            sys.stdout.flush()

    def close(self):
        pass

def dedupe_series_index(series: pd.Series) -> pd.Series:
    return series[~series.index.duplicated(keep='first')]


# --- Metrics Calculation ---
def calculate_metrics_for_schedule(env_instance_for_metrics, schedule_ref, first_job_scheduled_time_ref, current_time_ref):
    """Calculates performance metrics for a schedule."""
    metrics = {}

    makespan_timedelta = pd.Timedelta(current_time_ref - (first_job_scheduled_time_ref if first_job_scheduled_time_ref else env_instance_for_metrics.initial_simulation_start_time))
    makespan_minutes = makespan_timedelta.total_seconds() / 60
    metrics['Total Makespan'] = makespan_minutes

    metrics['Machine Utilization'] = {}
    machine_actual_processing_time = {m_id: 0.0 for m_id in env_instance_for_metrics.machine_list}
    for op in schedule_ref:
        machine_actual_processing_time[op['assigned_machine']] += op['process_time']

    for m_id in env_instance_for_metrics.machine_list:
        total_busy_time = machine_actual_processing_time.get(m_id, 0.0)
        if makespan_minutes > 0:
            metrics['Machine Utilization'][m_id] = (total_busy_time / makespan_minutes) * 100
        else:
            metrics['Machine Utilization'][m_id] = 0.0

    metrics['Operator Utilization'] = {}
    operator_actual_processing_time = {o_id: 0.0 for o_id in env_instance_for_metrics.operator_list}
    for op in schedule_ref:
        operator_actual_processing_time[op['assigned_operator']] += op['process_time']

    for o_id in env_instance_for_metrics.operator_list:
        total_busy_time = operator_actual_processing_time.get(o_id, 0.0)
        if makespan_minutes > 0:
            metrics['Operator Utilization'][o_id] = (total_busy_time / makespan_minutes) * 100
        else:
            metrics['Operator Utilization'][o_id] = 0.0

    recalculated_total_tardiness = 0.0
    recalculated_jobs_on_time_count = 0
    recalculated_jobs_late_count = 0
    completed_original_job_ids = set()

    original_order_ids_in_schedule = {op['original_order_id'] for op in schedule_ref}

    for original_order_id in original_order_ids_in_schedule:
        num_batches_expected = env_instance_for_metrics.original_job_num_batches.get(original_order_id, 0)
        batches_in_schedule = {op['batch_id'] for op in schedule_ref if op['original_order_id'] == original_order_id}

        if len(batches_in_schedule) == num_batches_expected:
            completed_original_job_ids.add(original_order_id)

            original_job_finish_time = max(op['finish_time'] for op in schedule_ref if op['original_order_id'] == original_order_id)

            original_job_data = env_instance_for_metrics.original_jobs_df[env_instance_for_metrics.original_jobs_df['Order ID'] == original_order_id].iloc[0]
            due_date_col = 'User Due Date'
            if due_date_col in original_job_data and pd.notna(original_job_data[due_date_col]):
                due_date = pd.to_datetime(original_job_data[due_date_col])
            else:
                due_date = pd.Timestamp('2025-12-31 00:00:00')

            delay_timedelta = pd.Timedelta(original_job_finish_time - due_date)
            delay_minutes = max(0, delay_timedelta.total_seconds() / 60)
            recalculated_total_tardiness += delay_minutes

            if delay_minutes == 0:
                recalculated_jobs_on_time_count += 1
            else:
                recalculated_jobs_late_count += 1

    total_original_jobs = len(env_instance_for_metrics.original_jobs_df)

    if len(completed_original_job_ids) > 0:
        metrics['Average Tardiness'] = recalculated_total_tardiness / len(completed_original_job_ids)
    else:
        metrics['Average Tardiness'] = 0.0

    metrics['Total Penalty Cost'] = env_instance_for_metrics.total_accumulated_penalty_cost

    metrics['Jobs On-Time Count'] = recalculated_jobs_on_time_count
    metrics['Jobs Late Count'] = recalculated_jobs_late_count
    metrics['Jobs Not Scheduled Count'] = total_original_jobs - len(completed_original_job_ids)

    if total_original_jobs > 0:
        metrics['Percentage On-Time'] = (recalculated_jobs_on_time_count / total_original_jobs) * 100
        metrics['Percentage Late'] = (recalculated_jobs_late_count / total_original_jobs) * 100
        metrics['Percentage Not Scheduled'] = (metrics['Jobs Not Scheduled Count'] / total_original_jobs) * 100
    else:
        metrics['Percentage On-Time'] = 0.0
        metrics['Percentage Late'] = 0.0
        metrics['Percentage Not Scheduled'] = 0.0

    return metrics

# --- DQN Agent & ReplayBuffer ---
class DQN(nn.Module):
    """The Deep Q-Network."""
    def __init__(self, obs_shape, n_actions):
        super(DQN, self).__init__()
        input_size = obs_shape[0]
        self.fc = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )

    def forward(self, x):
        return self.fc(x)

class ReplayBuffer:
    """Stores past experiences for training."""
    def __init__(self, max_size):
        self.buffer = deque(maxlen=max_size)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        if len(self.buffer) < batch_size:
            raise ValueError(f"Not enough samples in buffer ({len(self.buffer)}) for batch size ({batch_size})")
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return np.stack(state), action, reward, np.stack(next_state), done

    def __len__(self):
        return len(self.buffer)

# --- Training Loop ---
def train_dqn(env, episodes=500, batch_size=64):
    """Trains the DQN agent."""
    global best_jobs_not_scheduled_count_data
    global best_jobs_on_time_count_data
    global best_makespan
    global best_schedule_data
    global best_job_completion_times_data
    global best_total_tardiness_minutes_data
    global best_jobs_late_count_data
    global best_total_accumulated_penalty_cost_data
    global best_first_job_scheduled_time_data
    global best_current_time_data
    global best_episode_num

    model = DQN(env.observation_space.shape, env.action_space.n).float()
    rewards_per_episode = []

    target_model = DQN(env.observation_space.shape, env.action_space.n).float()
    target_model.load_state_dict(model.state_dict())
    target_model.eval()

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    memory = ReplayBuffer(max_size=10000)

    gamma = 0.99
    epsilon_start = 0.9
    epsilon_end = 0.01
    epsilon_decay_episodes = int(episodes * 0.8)
    target_update_frequency = 5

    def select_action(state, eps, env_instance):
        """Selects an action based on epsilon-greedy policy."""
        all_possible_actions = []

        for batch_idx in range(env_instance.num_jobs):
            batch_data = env_instance.jobs.iloc[batch_idx]
            batch_release_time = pd.to_datetime(batch_data.get("Job Start Date", env_instance.initial_simulation_start_time))
            if env_instance.current_time < batch_release_time:
                continue

            if batch_idx not in env_instance.remaining_jobs:
                continue

            if batch_idx in env_instance.job_committed_option:
                committed_option_idx = env_instance.job_committed_option[batch_idx]
                action_value = batch_idx * env_instance.num_options_per_job + committed_option_idx
                if env_instance._has_failed_too_many_times(batch_idx, committed_option_idx):
                    continue
                if env_instance._is_action_feasible(batch_idx, committed_option_idx, env_instance.current_time):
                    all_possible_actions.append(action_value)
            else:
                for opt_idx in range(env_instance.num_options_per_job):
                    action_value = batch_idx * env_instance.num_options_per_job + opt_idx
                    if env_instance._has_failed_too_many_times(batch_idx, opt_idx):
                        continue
                    if env_instance._is_action_feasible(batch_idx, opt_idx, env_instance.current_time):
                        all_possible_actions.append(action_value)

        if not all_possible_actions:
            return -1

        if random.random() < eps:
            chosen_action = random.choice(all_possible_actions)
            return chosen_action
        else:
            with torch.no_grad():
                state_tensor = torch.tensor(state).float().unsqueeze(0)
                q_values = model(state_tensor)
                masked_q_values = q_values.clone().squeeze(0)

                for action_idx in range(env_instance.action_space.n):
                    if action_idx not in all_possible_actions:
                        masked_q_values[action_idx] = -float('inf')

                if torch.all(masked_q_values == -float('inf')):
                    chosen_action = random.choice(all_possible_actions)
                    return chosen_action

                chosen_action = torch.argmax(masked_q_values).item()
                return chosen_action

    for ep in range(episodes):
        sys.stdout.flush()

        state = env.reset()
        total_reward = 0
        done = False
        step_count = 0

        eps = epsilon_start - (epsilon_start - epsilon_end) * min(1, ep / epsilon_decay_episodes)
        #eps=0.9
        # if ep < 500:
        #     eps = 0.9
        # elif ep < 600:
        #     eps = 0.7
        # elif ep < 700:
        #     eps = 0.5
        # elif ep < 800:
        #     eps = 0.3
        # else:
        #     eps = 0.1

        while not done:
            action = select_action(state, eps, env)

            if step_count > MAX_STEPS_PER_EPISODE:
                print(f"Episode {ep} exceeded max steps. Forcing termination.", file=sys.stdout)
                sys.stdout.flush()
                done = True
                total_reward -= 5000
                break

            if action == -1:
                next_state, reward, done, _ = env.step(action)
            else:
                try:
                    next_state, reward, done, _ = env.step(action)
                except Exception as e:
                    print(f"Episode {ep}, Step {step_count}: Error during env.step({action}): {e}", file=sys.stderr)
                    sys.stderr.flush()
                    total_reward -= 1000
                    done = True
                    break

            memory.push(state, action, reward, next_state, done)

            state = next_state
            total_reward += reward
            step_count += 1

            if len(memory) >= batch_size:
                try:
                    states_batch, actions_batch, rewards_batch, next_states_batch, dones_batch = memory.sample(batch_size)
                except ValueError:
                    continue

                valid_indices = [i for i, a in enumerate(actions_batch) if a != -1]

                if not valid_indices:
                    continue

                filtered_states = np.array([states_batch[i] for i in valid_indices])
                filtered_actions = [actions_batch[i] for i in valid_indices]
                filtered_rewards = [rewards_batch[i] for i in valid_indices]
                filtered_next_states = np.array([next_states_batch[i] for i in valid_indices])
                filtered_dones = [dones_batch[i] for i in valid_indices]

                states = torch.tensor(filtered_states).float()
                actions = torch.tensor(filtered_actions).unsqueeze(1)
                rewards_ = torch.tensor(filtered_rewards).float()
                next_states = torch.tensor(filtered_next_states).float()
                dones = torch.tensor(filtered_dones)

                q_vals = model(states).gather(1, actions).squeeze(1)

                next_action_choices = model(next_states).argmax(1).unsqueeze(1)
                next_q_vals = target_model(next_states).gather(1, next_action_choices).squeeze(1).detach()

                targets = rewards_ + gamma * next_q_vals * (~dones)

                loss = nn.MSELoss()(q_vals, targets)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        current_metrics = calculate_metrics_for_schedule(
            env, env.schedule, env.first_job_scheduled_time, env.current_time
        )
        current_makespan = current_metrics['Total Makespan']
        current_on_time_jobs = current_metrics['Jobs On-Time Count']
        current_jobs_not_scheduled = current_metrics['Jobs Not Scheduled Count']

        is_new_best = False

        if current_jobs_not_scheduled < best_jobs_not_scheduled_count_data:
            is_new_best = True
        elif current_jobs_not_scheduled == best_jobs_not_scheduled_count_data:
            if current_on_time_jobs > best_jobs_on_time_count_data:
                is_new_best = True
            elif current_on_time_jobs == best_jobs_on_time_count_data:
                if current_makespan < best_makespan:
                    is_new_best = True

        if is_new_best:
            best_jobs_not_scheduled_count_data = current_jobs_not_scheduled
            best_jobs_on_time_count_data = current_on_time_jobs
            best_makespan = current_makespan

            best_schedule_data = copy.deepcopy(env.schedule)
            best_job_completion_times_data = copy.deepcopy(env.original_job_completion_times)
            best_total_tardiness_minutes_data = env.total_tardiness_minutes
            best_jobs_late_count_data = current_metrics['Jobs Late Count']
            best_total_accumulated_penalty_cost_data = env.total_accumulated_penalty_cost
            best_first_job_scheduled_time_data = env.first_job_scheduled_time
            best_current_time_data = env.current_time
            best_episode_num = ep
            print(f"Episode {ep}: NEW BEST Schedule - Unscheduled: {best_jobs_not_scheduled_count_data}, On-Time: {best_jobs_on_time_count_data}, Makespan: {best_makespan:.2f}", file=sys.stdout)
            sys.stdout.flush()

        if ep % 50 == 0:
            print(f"Episode {ep}: Total Reward: {total_reward:.2f}, Epsilon: {eps:.2f}, Makespan: {current_makespan:.2f}", file=sys.stdout)
            sys.stdout.flush()

        rewards_per_episode.append(total_reward)

        if ep % target_update_frequency == 0:
            target_model.load_state_dict(model.state_dict())

    return model, rewards_per_episode, best_schedule_data, best_job_completion_times_data, \
           best_total_tardiness_minutes_data, best_jobs_on_time_count_data, \
           best_jobs_late_count_data, best_total_accumulated_penalty_cost_data, \
           best_first_job_scheduled_time_data, best_current_time_data, best_episode_num

# --- Gantt Chart Visualization ---
def plot_gantt(schedule, machine_list, operator_list, fixture_list, planned_machine_downtime, planned_operator_downtime, original_jobs_df_ref):
    """Plots a Gantt chart of the schedule."""
    if not schedule:
        print("No schedule generated to plot.", file=sys.stdout)
        sys.stdout.flush()
        return

    df_schedule = pd.DataFrame(schedule)

    scheduled_machines = df_schedule['assigned_machine'].dropna().unique().tolist()
    scheduled_operators = df_schedule['assigned_operator'].dropna().unique().tolist()
    scheduled_fixtures = df_schedule['assigned_fixture'].dropna().unique().tolist()

    y_labels = []
    y_positions = []
    current_y = 0

    y_labels.append('--- MACHINES ---')
    y_positions.append(current_y)
    current_y += 1
    machine_y_map = {}
    for m_id in sorted(machine_list):
        y_labels.append(m_id)
        y_positions.append(current_y)
        machine_y_map[m_id] = current_y
        current_y += 1

    y_labels.append('--- OPERATORS ---')
    y_positions.append(current_y)
    current_y += 1
    operator_y_map = {}
    for o_id in sorted(operator_list):
        y_labels.append(o_id)
        y_positions.append(current_y)
        operator_y_map[o_id] = current_y
        current_y += 1

    y_labels.append('--- FIXTURES ---')
    y_positions.append(current_y)
    current_y += 1
    fixture_y_map = {}
    for f_id in sorted(fixture_list):
        y_labels.append(f_id)
        y_positions.append(current_y)
        fixture_y_map[f_id] = current_y
        current_y += 1

    fig, ax = plt.subplots(figsize=(18, max(10, len(y_labels) * 0.7)))

    unique_original_order_ids = original_jobs_df_ref['Order ID'].unique()
    num_unique_original_jobs = len(unique_original_order_ids)
    job_color_map = {order_id: plt.cm.get_cmap('tab20', num_unique_original_jobs)(i)
                     for i, order_id in enumerate(sorted(unique_original_order_ids))}

    earliest_scheduled_time = df_schedule['start_time'].min() if not df_schedule.empty else pd.Timestamp('2024-07-01 00:00:00')

    # Plot planned downtimes
    for i, downtime in enumerate(planned_machine_downtime):
        resource_id = downtime['resource_id']
        dt_start = pd.to_datetime(downtime['start_time'])
        dt_end = pd.to_datetime(downtime['end_time'])
        if resource_id in machine_y_map:
            y_pos = machine_y_map[resource_id]
            downtime_start_timedelta = pd.Timedelta(dt_start - earliest_scheduled_time)
            downtime_end_timedelta = pd.Timedelta(dt_end - earliest_scheduled_time)
            downtime_start_min = downtime_start_timedelta.total_seconds() / 60
            downtime_end_min = downtime_end_timedelta.total_seconds() / 60
            if downtime_end_min > 0:
                ax.axvspan(max(0, downtime_start_min), downtime_end_min,
                           ymin=(y_pos - 0.3) / (len(y_labels) + 1),
                           ymax=(y_pos + 0.3) / (len(y_labels) + 1),
                           color='gray', alpha=0.3, label='Planned Downtime' if i == 0 else "")

    for i, downtime in enumerate(planned_operator_downtime):
        resource_id = downtime['resource_id']
        dt_start = pd.to_datetime(downtime['start_time'])
        dt_end = pd.to_datetime(downtime['end_time'])
        if resource_id in operator_y_map:
            y_pos = operator_y_map[resource_id]
            downtime_start_timedelta = pd.Timedelta(dt_start - earliest_scheduled_time)
            downtime_end_timedelta = pd.Timedelta(dt_end - earliest_scheduled_time)
            downtime_start_min = downtime_start_timedelta.total_seconds() / 60
            downtime_end_min = downtime_end_timedelta.total_seconds() / 60
            if downtime_end_min > 0:
                ax.axvspan(max(0, downtime_start_min), downtime_end_min,
                           ymin=(y_pos - 0.3) / (len(y_labels) + 1),
                           ymax=(y_pos + 0.3) / (len(y_labels) + 1),
                           color='gray', alpha=0.3, label='Planned Downtime' if i == 0 and len(planned_machine_downtime) == 0 else "")

    # Plot scheduled operations
    for _, entry in df_schedule.iterrows():
        original_order_id = entry['original_order_id']
        batch_id = entry['batch_id']
        original_part_id = entry['original_part_id']
        operation_name = entry['operation_name']
        assigned_machine = entry['assigned_machine']
        assigned_operator = entry['assigned_operator']
        assigned_fixture = entry['assigned_fixture']
        process_time = entry['process_time']
        start_time = entry['start_time']
        finish_time = entry['finish_time']
        option_idx = entry['option_idx']

        start_time_timedelta = pd.Timedelta(start_time - earliest_scheduled_time)
        duration_timedelta = pd.Timedelta(finish_time - start_time)
        start_time_minutes = start_time_timedelta.total_seconds() / 60
        duration_minutes = duration_timedelta.total_seconds() / 60

        task_label = f'{batch_id} (P{original_part_id}, Opt{option_idx + 1}): {operation_name}'
        color = job_color_map.get(original_order_id, 'black')

        if assigned_machine in machine_y_map:
            y_pos = machine_y_map[assigned_machine]
            ax.barh(y_pos, duration_minutes, left=start_time_minutes, height=0.6,
                    color=color, edgecolor='black', alpha=0.7)
            ax.text(start_time_minutes + duration_minutes/2, y_pos, task_label,
                    ha='center', va='center', color='black', fontsize=7, weight='bold')

        if assigned_operator in operator_y_map:
            y_pos = operator_y_map[assigned_operator]
            ax.barh(y_pos, duration_minutes, left=start_time_minutes, height=0.6,
                    color=color, edgecolor='black', alpha=0.7)
            ax.text(start_time_minutes + duration_minutes/2, y_pos, task_label,
                    ha='center', va='center', color='black', fontsize=7, weight='bold')

        if assigned_fixture in fixture_y_map:
            y_pos = fixture_y_map[assigned_fixture]
            ax.barh(y_pos, duration_minutes, left=start_time_minutes, height=0.6,
                    color=color, edgecolor='black', alpha=0.7)
            ax.text(start_time_minutes + duration_minutes/2, y_pos, task_label,
                    ha='center', va='center', color='black', fontsize=7, weight='bold')

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)

    ax.set_xlabel('Time (minutes relative to first scheduled operation)')
    ax.set_ylabel('Resources')
    ax.set_title('Gantt Chart - Shop Floor Schedule with Resource Utilization and Planned Downtime')
    ax.grid(axis='x', linestyle='--', alpha=0.7)

    if not df_schedule.empty:
        max_finish_time_timedelta_relative = pd.Timedelta(df_schedule['finish_time'].max() - earliest_scheduled_time)
        max_finish_time_minutes_relative = max_finish_time_timedelta_relative.total_seconds() / 60
        ax.set_xlim(0, max_finish_time_minutes_relative * 1.1)
    else:
        ax.set_xlim(0, 100)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left')

    plt.tight_layout()
    plt.show()

# --- Run Training & Test ---
env = ShopFloorEnv()

print("Starting DQN agent training!", file=sys.stdout)
sys.stdout.flush()

rewards = []
model = None

try:
    model, rewards, \
    best_schedule_data, best_job_completion_times_data, \
    best_total_tardiness_minutes_data, best_jobs_on_time_count_data, \
    best_jobs_late_count_data, best_total_accumulated_penalty_cost_data, \
    best_first_job_scheduled_time_data, best_current_time_data, best_episode_num = train_dqn(env, episodes=500, batch_size=64)
    print("Training completed.", file=sys.stdout)
    sys.stdout.flush()
except Exception as e:
    print(f"Error during training: {e}", file=sys.stderr)
    sys.stderr.flush()

print("Training finished.", file=sys.stdout)
sys.stdout.flush()

plt.figure(figsize=(10, 5))
plt.plot(rewards)
plt.title('DQN Training Rewards Over Episodes')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.grid(True)
plt.show()

print("\n--- Test Run with Trained Model! ---", file=sys.stdout)
sys.stdout.flush()

env_test = ShopFloorEnv(debug_feasibility_checks=True)
state = env_test.reset()
done = False
test_total_reward = 0
test_epsilon = 0.01

if model is not None:
    model.eval()
else:
    print("WARNING: Model not trained. Skipping test run.", file=sys.stderr)
    sys.stderr.flush()
    sys.exit(1)

def test_select_action(state, eps, env_instance, trained_model):
    """Action selection for testing."""
    all_possible_actions = []
    for batch_idx in range(env_instance.num_jobs):
        batch_data = env_instance.jobs.iloc[batch_idx]
        batch_release_time = pd.to_datetime(batch_data.get("Job Start Date", env_instance.initial_simulation_start_time))
        if env_instance.current_time < batch_release_time:
            continue

        if batch_idx not in env_instance.remaining_jobs:
            continue

        if batch_idx in env_instance.job_committed_option:
            committed_option_idx = env_instance.job_committed_option[batch_idx]
            action_value = batch_idx * env_instance.num_options_per_job + committed_option_idx
            if env_instance._has_failed_too_many_times(batch_idx, committed_option_idx):
                continue
            if env_instance._is_action_feasible(batch_idx, committed_option_idx, env_instance.current_time):
                all_possible_actions.append(action_value)
        else:
            for opt_idx in range(env_instance.num_options_per_job):
                action_value = batch_idx * env_instance.num_options_per_job + opt_idx
                if env_instance._has_failed_too_many_times(batch_idx, opt_idx):
                    continue
                if env_instance._is_action_feasible(batch_idx, opt_idx, env_instance.current_time):
                    all_possible_actions.append(action_value)

    if not all_possible_actions:
        return -1

    if random.random() < eps:
        chosen_action = random.choice(all_possible_actions)
        return chosen_action
    else:
        with torch.no_grad():
            state_tensor = torch.tensor(state).float().unsqueeze(0)
            q_values = trained_model(state_tensor)
            masked_q_values = q_values.clone().squeeze(0)
            for action_idx in range(env_instance.action_space.n):
                if action_idx not in all_possible_actions:
                    masked_q_values[action_idx] = -float('inf')
            if torch.all(masked_q_values == -float('inf')):
                chosen_action = random.choice(all_possible_actions)
                return chosen_action
            chosen_action = torch.argmax(masked_q_values).item()
            return chosen_action

test_step_count = 0
while not done:
    if test_step_count > MAX_STEPS_PER_EPISODE:
        print(f"Test Run exceeded max steps. Forcing termination.", file=sys.stdout)
        sys.stdout.flush()
        break

    action = test_select_action(state, test_epsilon, env_test, model)

    if action == -1:
        next_state, reward, done, _ = env_test.step(action)
    else:
        next_state, reward, done, _ = env_test.step(action)

    test_total_reward += reward
    state = next_state
    test_step_count += 1

print(f"Test Run Complete! Total Reward: {test_total_reward:.2f}", file=sys.stdout)
sys.stdout.flush()

env_test.render()

print("\n--- Schedule Quality Metrics for LAST Episode ---", file=sys.stdout)
sys.stdout.flush()

metrics_last_episode = calculate_metrics_for_schedule(
    env_test, env_test.schedule, env_test.first_job_scheduled_time, env_test.current_time
)

print(f"Total Makespan: {metrics_last_episode['Total Makespan']:.2f} minutes", file=sys.stdout)
print("\nMachine Utilization Rate (%):", file=sys.stdout)
for machine, util in metrics_last_episode['Machine Utilization'].items():
    print(f"   {machine}: {util:.2f}%", file=sys.stdout)
print("\nOperator Utilization Rate (%):", file=sys.stdout)
for operator, util in metrics_last_episode['Operator Utilization'].items():
    print(f"   {operator}: {util:.2f}%", file=sys.stdout)
print(f"\nAverage Tardiness: {metrics_last_episode['Average Tardiness']:.2f} minutes/job", file=sys.stdout)
print(f"Total Penalty Cost (accumulated): {metrics_last_episode['Total Penalty Cost']:.2f}", file=sys.stdout)
print("\nJobs Completed Status:", file=sys.stdout)
print(f"   On-Time: {metrics_last_episode['Jobs On-Time Count']} jobs ({metrics_last_episode['Percentage On-Time']:.2f}%)", file=sys.stdout)
print(f"   Late: {metrics_last_episode['Jobs Late Count']} jobs ({metrics_last_episode['Percentage Late']:.2f}%)", file=sys.stdout)
print(f"   Not Scheduled: {metrics_last_episode['Jobs Not Scheduled Count']} jobs ({metrics_last_episode['Percentage Not Scheduled']:.2f}%)", file=sys.stdout)
sys.stdout.flush()

print("\n--- Generating Gantt Chart for LAST Episode ---", file=sys.stdout)
sys.stdout.flush()

plot_gantt(env_test.schedule, env_test.machine_list, env_test.operator_list, env_test.fixture_list, env_test.planned_machine_downtime, env_test.planned_operator_downtime, env_test.original_jobs_df)

# --- Display Best Schedule ---
if best_schedule_data is not None:
    print(f"\n--- Best Schedule Achieved (Episode {best_episode_num}) ---", file=sys.stdout)
    print(f"  Jobs Not Scheduled: {best_jobs_not_scheduled_count_data}", file=sys.stdout)
    print(f"  Jobs On-Time: {best_jobs_on_time_count_data}", file=sys.stdout)
    print(f"  Total Makespan: {best_makespan:.2f} minutes", file=sys.stdout)
    sys.stdout.flush()

    print("\n--- Scheduled Operations (Best Schedule) ---", file=sys.stdout)
    sys.stdout.flush()

    # Prepare clean list for CSV/Excel
    clean_schedule_list = []

    for entry in best_schedule_data:
        start_str = entry['start_time'].strftime('%Y-%m-%d %H:%M') if pd.notna(entry['start_time']) else 'N/A'
        finish_str = entry['finish_time'].strftime('%Y-%m-%d %H:%M') if pd.notna(entry['finish_time']) else 'N/A'
        option_number = entry['option_idx'] + 1 if 'option_idx' in entry else 1

        # Print nicely
        print(
            f"Batch {entry['batch_id']} (Original Order {entry['original_order_id']}, Part {entry['original_part_id']}) "
            f"Op {entry['operation_name']} (Option {option_number}): "
            f"Machine={entry['assigned_machine']}, Operator={entry['assigned_operator']}, "
            f"Fixture={entry['assigned_fixture']}, PT={entry['process_time']:.2f} min, "
            f"Start={start_str}, Finish={finish_str}",
            file=sys.stdout
        )

        # Flatten entry for DataFrame
        clean_entry = {
            'Batch ID': entry['batch_id'],
            'Original Order ID': entry['original_order_id'],
            'Part ID': entry['original_part_id'],
            'Operation': entry['operation_name'],
            'Option': option_number,
            'Machine': entry['assigned_machine'],
            'Operator': entry['assigned_operator'],
            'Fixture': entry['assigned_fixture'],
            'Process Time (min)': entry['process_time'],
            'Start Time': start_str,
            'Finish Time': finish_str
        }
        clean_schedule_list.append(clean_entry)

    sys.stdout.flush()
    print(f"Total operations scheduled: {len(best_schedule_data)}", file=sys.stdout)
    print(f"Final simulation time (for best schedule): {best_current_time_data.strftime('%Y-%m-%d %H:%M')}", file=sys.stdout)
    sys.stdout.flush()

    # Metrics
    print("\n--- Schedule Quality Metrics for BEST Schedule Episode ---", file=sys.stdout)
    sys.stdout.flush()

    metrics_best_schedule = calculate_metrics_for_schedule(
        env, best_schedule_data, best_first_job_scheduled_time_data, best_current_time_data
    )

    print(f"Total Makespan: {metrics_best_schedule['Total Makespan']:.2f} minutes", file=sys.stdout)
    print("\nMachine Utilization Rate (%):", file=sys.stdout)
    for machine, util in metrics_best_schedule['Machine Utilization'].items():
        print(f"   {machine}: {util:.2f}%", file=sys.stdout)

    print("\nOperator Utilization Rate (%):", file=sys.stdout)
    for operator, util in metrics_best_schedule['Operator Utilization'].items():
        print(f"   {operator}: {util:.2f}%", file=sys.stdout)

    print(f"\nAverage Tardiness: {metrics_best_schedule['Average Tardiness']:.2f} minutes/job", file=sys.stdout)
    print(f"Total Penalty Cost (accumulated): {metrics_best_schedule['Total Penalty Cost']:.2f}", file=sys.stdout)
    print("\nJobs Completed Status:", file=sys.stdout)
    print(f"   On-Time: {metrics_best_schedule['Jobs On-Time Count']} jobs ({metrics_best_schedule['Percentage On-Time']:.2f}%)", file=sys.stdout)
    print(f"   Late: {metrics_best_schedule['Jobs Late Count']} jobs ({metrics_best_schedule['Percentage Late']:.2f}%)", file=sys.stdout)
    print(f"   Not Scheduled: {metrics_best_schedule['Jobs Not Scheduled Count']} jobs ({metrics_best_schedule['Percentage Not Scheduled']:.2f}%)", file=sys.stdout)
    sys.stdout.flush()

    # Gantt chart
    print("\n--- Generating Gantt Chart for BEST Schedule Episode ---", file=sys.stdout)
    sys.stdout.flush()
    plot_gantt(
        best_schedule_data,
        env.machine_list,
        env.operator_list,
        env.fixture_list,
        env.planned_machine_downtime,
        env.planned_operator_downtime,
        env.original_jobs_df
    )

    # Export to CSV/Excel
    df_best_schedule = pd.DataFrame(clean_schedule_list)
    df_best_schedule.to_csv("best_schedule.csv", index=False)
    df_best_schedule.to_excel("best_schedule.xlsx", index=False)
    print("\n✅ Best schedule has been exported to 'best_schedule.csv' and 'best_schedule.xlsx'", file=sys.stdout)

else:
    print("No best schedule data found, likely due to an error during training.", file=sys.stdout)
    sys.stdout.flush()

env.close()


